## Data Cleaning: Netflix Titles

Source: Kaggle, shivamb/netflix-shows (netflix_titles.csv).

In [1]:
import pandas as pd

df = pd.read_csv("../data/netflix_titles.csv")
print(df.shape)
df.head()

(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [2]:
df.info()

print("\nMissing values per column:")
print(df.isna().sum())

print("\nFully duplicated rows:", df.duplicated().sum())
print("Duplicated show_id values:", df["show_id"].duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB

Missing values per column:
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
d

In [3]:
# Find rows where the duration was typed into the rating column
shifted = df["rating"].str.contains("min", na=False)
print("Shifted rows found:", shifted.sum())

# Move the value into duration, then blank out the rating
df.loc[shifted, "duration"] = df.loc[shifted, "rating"]
df.loc[shifted, "rating"] = pd.NA

df.loc[shifted, ["title", "rating", "duration"]]

Shifted rows found: 3


,title,rating,duration
5541,Louis C.K. 2017,NaN,74 min
5794,Louis C.K.: Hilarious,NaN,84 min
5813,Louis C.K.: Live at the Comedy Store,NaN,66 min


In [5]:
#convert date_added to real dates
df["date_added"] = df["date_added"].str.strip()
df["date_added"] = pd.to_datetime(df["date_added"], format="%B %d, %Y")

df["date_added"].info()
df["date_added"].isna().sum()

<class 'pandas.Series'>
RangeIndex: 8807 entries, 0 to 8806
Series name: date_added
Non-Null Count  Dtype         
--------------  -----         
8797 non-null   datetime64[us]
dtypes: datetime64[us](1)
memory usage: 68.9 KB


np.int64(10)

In [6]:
#Fill missing director, cast and country
for col in ["director", "cast", "country"]:
    df[col] = df[col].fillna("Unknown")

print(df[["director", "cast", "country"]].isna().sum())
print((df["country"] == "Unknown").sum())

director    0
cast        0
country     0
dtype: int64
831


In [7]:
# Validate the cleaned date
print("Shape:", df.shape)
print("\nMissing values per column:")
print(df.isna().sum())

print("\nDuplicated show_id values:", df["show_id"].duplicated().sum())
print("Ratings containing 'min':", df["rating"].str.contains("min", na=False).sum())
print("date_added dtype:", df["date_added"].dtype)
print("Unknown countries:", (df["country"] == "Unknown").sum())

Shape: (8807, 12)

Missing values per column:
show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           7
duration         0
listed_in        0
description      0
dtype: int64

Duplicated show_id values: 0
Ratings containing 'min': 0
date_added dtype: datetime64[us]
Unknown countries: 831


In [10]:
# Split duration into 2 numeric columns 
df["duration_min"] = pd.to_numeric(df["duration"].str.extract(r"(\d+) min")[0])
df["seasons"] = pd.to_numeric(df["duration"].str.extract(r"(\d+) Season")[0])

print(df.shape)
print("duration_min filled:", df["duration_min"].notna().sum())
print("seasons filled:", df["seasons"].notna().sum())
print(df.groupby("type")[["duration_min", "seasons"]].count())

(8807, 14)
duration_min filled: 6131
seasons filled: 2676
         duration_min  seasons
type                          
Movie            6131        0
TV Show             0     2676


In [11]:
#export data
import os

os.makedirs("../data/cleaned", exist_ok=True)
df.to_csv("../data/cleaned/netflix_cleaned.csv", index=False)
print("Saved.")

Saved.
